# Adding noise channels makes a model worse

The experiment that shows generalization is not something you get for free from more features. Same model, same digits, plus 784 columns of pure noise.

**Runs on:** CPU — about 3 minutes &nbsp;·&nbsp; **Slides:** [Chapter 5 — Fundamentals of Machine Learning](../../../course-web-slides/ch05/index.html) &nbsp;·&nbsp; **Section:** 01 — Generalization: the goal of machine learning

---

## Two versions of MNIST

In [ ]:
import numpy as np
from keras.datasets import mnist

(train_images, train_labels), _ = mnist.load_data()
train_images = train_images.reshape((60000, 28 * 28)).astype("float32") / 255

train_images_with_noise_channels = np.concatenate(
    [train_images, np.random.random((len(train_images), 784))], axis=1)

train_images_with_zeros_channels = np.concatenate(
    [train_images, np.zeros((len(train_images), 784))], axis=1)

print(train_images_with_noise_channels.shape)

Both new versions are 1568 columns wide. One has random noise in the extra half; the other has zeros. **Neither carries any information about the digit.**

## Training both

In [ ]:
import keras
from keras import layers

def get_model():
    keras.utils.set_random_seed(0)
    model = keras.Sequential([
        layers.Dense(512, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="rmsprop",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

history_noise = get_model().fit(
    train_images_with_noise_channels, train_labels,
    epochs=10, batch_size=128, validation_split=0.2, verbose=0)

history_zeros = get_model().fit(
    train_images_with_zeros_channels, train_labels,
    epochs=10, batch_size=128, validation_split=0.2, verbose=0)

## The result

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, 11)
plt.figure(figsize=(7, 4.4))
plt.plot(epochs, history_noise.history["val_accuracy"], "b-",
         label="noise channels")
plt.plot(epochs, history_zeros.history["val_accuracy"], "b--",
         label="zero channels")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend()
plt.title("Effect of noise channels on validation accuracy")
plt.show()

gap = (history_zeros.history["val_accuracy"][-1]
       - history_noise.history["val_accuracy"][-1])
print(f"final gap: {gap:.3f}")

Expected output:

```
final gap: about 0.01 to 0.02 — one to two percentage points
```

Both sets of extra columns are uninformative. Only one **hurts**.

The difference is that noise offers something to latch onto: with enough capacity the model finds correlations in the random columns that happen to hold on the training set and hold nowhere else. ==Feature selection is not tidiness; it is a defence.==

## How much noise before it collapses

In [ ]:
results = {}
for n_noise in [0, 128, 784, 2000]:
    x = train_images if n_noise == 0 else np.concatenate(
        [train_images, np.random.random((len(train_images), n_noise))], axis=1)
    h = get_model().fit(x, train_labels, epochs=6, batch_size=128,
                        validation_split=0.2, verbose=0)
    results[n_noise] = h.history["val_accuracy"][-1]
    print(f"{n_noise:5d} noise columns -> val acc {results[n_noise]:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(list(results), list(results.values()), "o-")
plt.xlabel("noise columns added"); plt.ylabel("validation accuracy")
plt.title("Degradation is gradual, not a cliff")
plt.show()

It degrades smoothly. There is no threshold to stay under — which is exactly why the practical rule is to measure feature usefulness rather than to guess at it.

---

## What to take away

- Uninformative features are not harmless: **noise hurts, zeros do not**.
- With enough capacity a model will find correlations in noise that hold only on the training set.
- Degradation is gradual, so there is no safe amount of junk to leave in.
- Feature selection is a generalization technique, not housekeeping.